# t-stack-trajectory — worked example 3: stack vs cat for trajectory assembly

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `t-stack-trajectory`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Both `torch.stack` and `torch.cat` can create trajectories, but they differ in what they require and what they produce. `stack` inserts a new axis and requires all tensors to have the same shape. `cat` concatenates along an existing axis and requires matching shapes on all axes except the concatenation axis. For trajectory assembly, `stack` is usually cleaner because per-step snapshots are naturally equal-shaped.

## Worked solution

**Step 1 — Using stack.**
If each snapshot has shape `(B, D)`, `t.stack(snapshots, dim=0)` gives `(T, B, D)`. The snapshot shape is preserved exactly.

**Step 2 — Using cat instead.**
To get the same result with `cat`, each snapshot must be unsqueezed first: `t.cat([s.unsqueeze(0) for s in snapshots], dim=0)`. This is more verbose and slower.

**Step 3 — When cat is better.**
If each snapshot is already rank-1 higher (e.g., shape `(1, B, D)`) for some reason, `cat` directly along `dim=0` avoids the extra squeeze.

**Step 4 — Verify both agree.**
The two resulting tensors should be element-wise equal. Use `torch.allclose` to confirm.

In [ ]:
import torch as t

t.manual_seed(7)
B, D, T = 3, 5, 4

# Create snapshots
t.manual_seed(7)
snapshots = [t.randn(B, D) for _ in range(T)]

# Method 1: stack
traj_stack = t.stack(snapshots, dim=0)       # (T, B, D)

# Method 2: cat with manual unsqueeze
traj_cat = t.cat([s.unsqueeze(0) for s in snapshots], dim=0)  # (T, B, D)

print('stack shape:', traj_stack.shape)   # (4, 3, 5)
print('cat shape:', traj_cat.shape)       # (4, 3, 5)
print('Both equal:', t.allclose(traj_stack, traj_cat))   # True

# Demonstrate access patterns
print('Step 2, batch 1 (stack):', traj_stack[2, 1])
print('Step 2, batch 1 (cat):', traj_cat[2, 1])